In [1]:
import os
import struct
from scipy import signal as sig
import numpy as np
from matplotlib import pyplot as plt

In [ ]:
PASSWORD = "12345678" #add your rpi password here

!echo $PASSWORD | sudo -S chmod 666 /dev/xdma0_h2c_0
!echo $PASSWORD | sudo -S chmod 666 /dev/xdma0_c2h_0
!echo $PASSWORD | sudo -S chmod 666 /dev/xdma0_user
!echo $PASSWORD | sudo -S chmod 666 /dev/xdma0_control

!ls -l /dev/xdma0_*

[sudo] password for ce-rpi: chmod: cannot access '/dev/xdma0_user': No such file or directory
crw-rw-rw- 1 root root 509, 36 Jun 18 14:12 /dev/xdma0_c2h_0
crw-rw-rw- 1 root root 509,  1 Jun 18 14:12 /dev/xdma0_control
crw------- 1 root root 509, 10 Jun 18 14:12 /dev/xdma0_events_0
crw------- 1 root root 509, 11 Jun 18 14:12 /dev/xdma0_events_1
crw------- 1 root root 509, 20 Jun 18 14:12 /dev/xdma0_events_10
crw------- 1 root root 509, 21 Jun 18 14:12 /dev/xdma0_events_11
crw------- 1 root root 509, 22 Jun 18 14:12 /dev/xdma0_events_12
crw------- 1 root root 509, 23 Jun 18 14:12 /dev/xdma0_events_13
crw------- 1 root root 509, 24 Jun 18 14:12 /dev/xdma0_events_14
crw------- 1 root root 509, 25 Jun 18 14:12 /dev/xdma0_events_15
crw------- 1 root root 509, 12 Jun 18 14:12 /dev/xdma0_events_2
crw------- 1 root root 509, 13 Jun 18 14:12 /dev/xdma0_events_3
crw------- 1 root root 509, 14 Jun 18 14:12 /dev/xdma0_events_4
crw------- 1 root root 509, 15 Jun 18 14:12 /dev/xdma0_events_5
crw-----

In [3]:
# Open devices for DMA operation on sudo
xdma_axis_rd_data = os.open('/dev/xdma0_c2h_0',os.O_RDONLY)
xdma_axis_wr_data = os.open('/dev/xdma0_h2c_0',os.O_WRONLY)

In [6]:
# Random 8x8 matrices with values 0-999999
A = np.random.randint(0, 100000000, size=(8,8), dtype=np.uint32)
B = np.random.randint(0, 100000000, size=(8,8), dtype=np.uint32)

print("A =")
print(A)

print("\nB =")
print(B)

# Expected result
C_ref = A @ B

print("\nExpected C =")
print(C_ref)

tx = np.concatenate([A.flatten(), B.flatten()])

data = struct.pack("<128I", *tx)

os.pwrite(xdma_axis_wr_data, data, 0)

A =
[[28870784 66771038 93816186 85759663 95755013 88777448   443615 86888729]
 [80537721 62007423 56160578 68522044 74525346 77161454 82572509 99246755]
 [94595327 21319620 23226416  6572093  3099094 21112953 11482755 96610291]
 [67077844 23364466 97263050 23375835  4749060 92223326 33955418   491212]
 [87282412 84044712 73114036 12332857 65807956 67779099 74571063 32801288]
 [84837701  7708190 21617888 69015233 43588587 73318689 91306153 99788749]
 [85135115 36353044 49948829 66033028 81466042 29733125  2869557 40800081]
 [73478674  3587745 82889966 89738078 69573058 70625952 82358553 82974253]]

B =
[[73489641 34733685 41914671 12336543 35558681 31389033 10471601   398524]
 [72429678 77960763 36715372  1290517 51345861 75363944 31513078 50100800]
 [54404364 75676796 61237927 84222582 93648401 45172215 69967838  8940658]
 [56208604 35594689  7771213  7922896 49888786 78817756 72224524 69697695]
 [32207215 76241380 28992790 19118893 82164683 99760408 99373253 15670186]
 [50675916 4986

512

In [7]:
rx_bytes = os.pread(xdma_axis_rd_data, 512, 0)

C = np.array(struct.unpack("<64I", rx_bytes)).reshape(8, 8)

print(C)

if np.array_equal(C, C_ref):
    print("\nResult is correct")
else:
    print("\nResult is incorrect")

[[3435970455 2351184422 2088728951  396251202 2251280844 4267445440
  3236894564 3807239197]
 [2064286581 1052034157 2974141557 3963457233 3781573541 2764870125
   488198580 1554039614]
 [3865912193 3010476305 3443245886 2860963756 2773863932 1460316249
  3905183513 1624186195]
 [2123334220  442157289 3730852303  466573472 3413400868 1292073240
  3676112520 2092705381]
 [  32519694 4270060509 3535132320 3991951487 2248989149 2406323953
  2487147908 3317471157]
 [3851634342  922540191  991683890 2534696975 2043387897 3447897123
  3564885698 3271709653]
 [2407865089 3476935862 1788384024 2744748700 1901268553 3062637024
  3419304177 2405383658]
 [4216834086 1052324900 4223827928 3817996942 2004070772  199501136
  1234694883 1348524048]]

Result is correct


In [1]:
#cpu benchmark 

import numpy as np
import time

NITER = 10000
N = 8

A = np.random.randint(0, 1000000, (N,N), dtype=np.uint32)
B = np.random.randint(0, 1000000, (N,N), dtype=np.uint32)

t0 = time.perf_counter_ns()

for _ in range(NITER):
    #naive matrix multiplication
    C = np.zeros((N,N), dtype=np.uint32)
    for i in range(N):
        for j in range(N):
            for k in range(N):
                C[i,j] += A[i,k] * B[k,j]
    

t1 = time.perf_counter_ns()

cpu_latency_us = (t1 - t0) / NITER / 1000

print(f"CPU latency: {cpu_latency_us:.3f} us")

/tmp/ipykernel_4400/417674553.py:20: RuntimeWarning: overflow encountered in scalar multiply
  C[i,j] += A[i,k] * B[k,j]
/tmp/ipykernel_4400/417674553.py:20: RuntimeWarning: overflow encountered in scalar add
  C[i,j] += A[i,k] * B[k,j]


CPU latency: 1157.524 us


In [4]:
import numpy as np
import struct
import time
import os

N = 8
NITER = 1000

A = np.random.randint(0,1000000,(N,N),dtype=np.uint32)
B = np.random.randint(0,1000000,(N,N),dtype=np.uint32)

tx = np.concatenate([A.flatten(), B.flatten()])
data = struct.pack(f"<{2*N*N}I", *tx)

# warmup
os.pwrite(xdma_axis_wr_data, data, 0)
os.pread(xdma_axis_rd_data, N*N*4, 0)

t0 = time.perf_counter_ns()

for _ in range(NITER):
    os.pwrite(xdma_axis_wr_data, data, 0)
    rx = os.pread(xdma_axis_rd_data, N*N*4, 0)

t1 = time.perf_counter_ns()

fpga_latency_us = (t1 - t0) / NITER / 1000

print(f"FPGA end-to-end latency: {fpga_latency_us:.3f} us")

FPGA end-to-end latency: 43.148 us
